In [ ]:
# ==========================================
# ISPY1 Data Cleaning
# ==========================================

import os
import numpy as np
import pandas as pd
import nibabel as nib

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Dataset paths
base_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"

dce_path = os.path.join(base_path, "dce")
mask_path = os.path.join(base_path, "mask")

metadata_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_spy1_metadata.csv"

print("DCE path:", dce_path)
print("Mask path:", mask_path)
print("Metadata path:", metadata_path)

In [ ]:
# Load file lists and metadata

dce_files = sorted([
    f for f in os.listdir(dce_path)
    if f.endswith(".nii.gz")
])

mask_files = sorted([
    f for f in os.listdir(mask_path)
    if f.endswith(".nii.gz")
])

metadata = pd.read_csv(metadata_path)

print("DCE volumes:", len(dce_files))
print("Masks:", len(mask_files))
print("Metadata rows:", len(metadata))

In [ ]:
# ==========================================
# Step 1: Build patient-level cohort
# ==========================================

# Extract patient IDs from DCE filenames
image_patient_ids = sorted({
    f.split("_")[1]
    for f in dce_files
})

# Add ISPY1_ prefix to match metadata
image_patient_ids = {
    f"ISPY1_{pid}"
    for pid in image_patient_ids
}

# Patient IDs from metadata
metadata_patient_ids = set(metadata["pid"])

# Find overlaps and differences
common_ids = image_patient_ids & metadata_patient_ids
image_only_ids = image_patient_ids - metadata_patient_ids
metadata_only_ids = metadata_patient_ids - image_patient_ids

print("Patients in imaging dataset:", len(image_patient_ids))
print("Patients in metadata:", len(metadata_patient_ids))
print("Patients present in both:", len(common_ids))
print("Imaging patients missing metadata:", len(image_only_ids))
print("Metadata patients without imaging:", len(metadata_only_ids))

In [ ]:
# ==========================================
# Step 2: Create patient-level master table
# ==========================================

patient_records = []

for patient_id in sorted(image_patient_ids):

    # Find this patient's DCE files
    patient_dce = sorted([
        f for f in dce_files
        if f"ISPY1_{patient_id.split('_')[1]}_" in f
    ])

    # Find this patient's mask
    patient_mask = [
        f for f in mask_files
        if f"ISPY1_{patient_id.split('_')[1]}_" in f
    ]

    # Metadata row
    patient_metadata = metadata[
        metadata["pid"] == patient_id
    ]

    # Create record
    record = {
        "pid": patient_id,
        "dce_0": os.path.join(dce_path, patient_dce[0]),
        "dce_1": os.path.join(dce_path, patient_dce[1]),
        "dce_2": os.path.join(dce_path, patient_dce[2]),
        "mask": os.path.join(mask_path, patient_mask[0])
    }

    # Add metadata
    if len(patient_metadata) == 1:
        for column in metadata.columns:
            if column != "pid":
                record[column] = patient_metadata.iloc[0][column]

    patient_records.append(record)


patient_df = pd.DataFrame(patient_records)

print("Patients in master table:", len(patient_df))
print("Columns:", len(patient_df.columns))

print("\nFirst 5 patients:")
display(patient_df.head())

In [ ]:
# ==========================================
# Verify all file paths in master table
# ==========================================

path_columns = ["dce_0", "dce_1", "dce_2", "mask"]

for column in path_columns:
    exists = patient_df[column].apply(os.path.exists)

    print(f"{column}:")
    print(f"  Files found: {exists.sum()}/{len(exists)}")
    print(f"  Missing: {(~exists).sum()}")


In [ ]:
# ==========================================
# Step 3: Validate tumor masks
# ==========================================

mask_quality = []

for _, row in patient_df.iterrows():

    mask_img = nib.load(row["mask"])
    mask_data = mask_img.get_fdata()

    unique_values = np.unique(mask_data)

    tumor_voxels = np.sum(mask_data > 0)

    mask_quality.append({
        "pid": row["pid"],
        "unique_values": unique_values,
        "tumor_voxels": tumor_voxels,
        "empty": tumor_voxels == 0
    })

mask_quality_df = pd.DataFrame(mask_quality)

print("Masks checked:", len(mask_quality_df))

print("\nEmpty masks:",
      mask_quality_df["empty"].sum())

print("\nTumor voxel statistics:")
print(
    mask_quality_df["tumor_voxels"].describe()
)

In [ ]:
# ==========================================
# Check whether all masks are binary
# ==========================================

non_binary_masks = []

for _, row in patient_df.iterrows():

    mask_img = nib.load(row["mask"])
    mask_data = mask_img.get_fdata()

    unique_values = np.unique(mask_data)

    # A valid binary mask should contain only 0 and 1
    if not np.all(np.isin(unique_values, [0, 1])):
        non_binary_masks.append(
            (row["pid"], unique_values)
        )

print("Masks checked:", len(patient_df))
print("Non-binary masks:", len(non_binary_masks))

if non_binary_masks:
    print("\nNon-binary masks:")
    for item in non_binary_masks:
        print(item)
else:
    print("All masks are binary (0 and 1).")

In [ ]:
# ==========================================
# Save validated patient-level dataset
# ==========================================

cleaned_path = os.path.join(
    "/content/drive/MyDrive/Breast_Cancer_AI_Project/data",
    "cleaned"
)

os.makedirs(cleaned_path, exist_ok=True)

master_table_path = os.path.join(
    cleaned_path,
    "patient_master.csv"
)

patient_df.to_csv(master_table_path, index=False)

print("Saved patient master table to:")
print(master_table_path)

print("\nPatients saved:", len(patient_df))

In [ ]:
# ==========================================
# Identify missing pCR patient
# ==========================================

missing_pcr = patient_df[
    patient_df["pCR"].isna()
]

print("Patients with missing pCR:", len(missing_pcr))

display(
    missing_pcr[[
        "pid",
        "pCR",
        "age",
        "ER",
        "PR",
        "HER2",
        "HR",
        "HR_HER2_STATUS"
    ]]
)

In [ ]:
# ==========================================
# Check prevalence of extreme intensities
# ==========================================

extreme_stats = []

for filename in dce_files:

    img = nib.load(os.path.join(dce_path, filename))
    data = img.get_fdata()

    total_voxels = data.size

    # Count very high intensity voxels
    count_gt_1000 = np.sum(data > 1000)
    count_gt_10000 = np.sum(data > 10000)
    count_gt_60000 = np.sum(data > 60000)

    extreme_stats.append({
        "filename": filename,
        "pct_gt_1000": (count_gt_1000 / total_voxels) * 100,
        "pct_gt_10000": (count_gt_10000 / total_voxels) * 100,
        "pct_gt_60000": (count_gt_60000 / total_voxels) * 100
    })

extreme_df = pd.DataFrame(extreme_stats)

print("Volumes checked:", len(extreme_df))

print("\nPercentage of voxels > 1000:")
print(extreme_df["pct_gt_1000"].describe())

print("\nPercentage of voxels > 10000:")
print(extreme_df["pct_gt_10000"].describe())

print("\nVolumes containing voxels > 60000:",
      (extreme_df["pct_gt_60000"] > 0).sum())

In [ ]:
# ==========================================
# Check background proportion in DCE volumes
# ==========================================

background_stats = []

for filename in dce_files:

    img = nib.load(os.path.join(dce_path, filename))
    data = img.get_fdata()

    total_voxels = data.size
    background_voxels = np.sum(data == 0)
    non_background_voxels = total_voxels - background_voxels

    background_stats.append({
        "filename": filename,
        "background_pct": (background_voxels / total_voxels) * 100,
        "non_background_pct": (non_background_voxels / total_voxels) * 100
    })

background_df = pd.DataFrame(background_stats)

print("Volumes checked:", len(background_df))

print("\nBackground percentage:")
print(background_df["background_pct"].describe())

print("\nNon-background percentage:")
print(background_df["non_background_pct"].describe())

In [ ]:
# ==========================================
# Check DCE-mask spatial alignment
# ==========================================

affine_mismatches = []

for _, row in patient_df.iterrows():

    # Load DCE 0 and mask
    dce_img = nib.load(row["dce_0"])
    mask_img = nib.load(row["mask"])

    # Compare affine matrices
    if not np.allclose(dce_img.affine, mask_img.affine):
        affine_mismatches.append(row["pid"])


print("Patients checked:", len(patient_df))
print("DCE-mask affine mismatches:", len(affine_mismatches))

if affine_mismatches:
    print("\nMismatched patients:")
    print(affine_mismatches)
else:
    print("All DCE volumes and masks are spatially aligned.")

In [ ]:
# ==========================================
# Check DCE acquisition metadata
# ==========================================

# Show columns related to DCE timing/acquisition
dce_metadata_columns = [
    col for col in metadata.columns
    if any(term in col.lower()
           for term in ["pre", "post", "time", "acq", "dcm"])
]

print("Relevant metadata columns:")
print(dce_metadata_columns)

print("\nMetadata values for these columns:")
display(metadata[dce_metadata_columns].head())

### DCE Acquisition Phases

The BreastDCEDL dataset defines three selected DCE time points:

1. Pre-contrast
2. Early post-contrast
3. Late post-contrast

The MinCrop dataset contains three DCE volumes per patient corresponding to these selected phases. For the MinCrop files, the acquisitions are ordered as:

- `dce_0` → pre-contrast
- `dce_1` → early post-contrast
- `dce_2` → late post-contrast

These three volumes are spatially aligned and will be treated as three corresponding DCE channels during downstream preprocessing/modeling.

In [ ]:
# ==========================================
# Preprocessing Step 1
# Test robust intensity normalization
# ==========================================

def normalize_dce(data):

    # Keep only non-background voxels
    nonzero = data[data > 0]

    # Robust intensity limits
    lower = np.percentile(nonzero, 1)
    upper = np.percentile(nonzero, 99)

    # Clip extreme values
    clipped = np.clip(data, lower, upper)

    # Normalize to 0-1
    normalized = (clipped - lower) / (upper - lower)

    # Restore background
    normalized[data == 0] = 0

    return normalized, lower, upper


# Test on patient 1219 DCE 0
test_file = patient_df.loc[
    patient_df["pid"] == "ISPY1_1219", "dce_0"
].iloc[0]

img = nib.load(test_file)
data = img.get_fdata()

normalized, lower, upper = normalize_dce(data)

print("Original shape:", data.shape)

print("\nOriginal intensity:")
print("Min:", data.min())
print("Max:", data.max())

print("\nNormalization limits:")
print("1st percentile:", lower)
print("99th percentile:", upper)

print("\nNormalized intensity:")
print("Min:", normalized.min())
print("Max:", normalized.max())

In [ ]:
# ==========================================
# Compare raw vs normalized MRI
# ==========================================
import matplotlib.pyplot as plt
z = data.shape[0] // 2

raw_slice = data[z]
norm_slice = normalized[z]

plt.figure(figsize=(7, 7))

plt.imshow(raw_slice, cmap="gray")
plt.title("Patient 1219 DCE 0 — Raw")
plt.axis("off")
plt.show()


plt.figure(figsize=(7, 7))

plt.imshow(norm_slice, cmap="gray", vmin=0, vmax=1)
plt.title("Patient 1219 DCE 0 — Normalized")
plt.axis("off")
plt.show()

In [ ]:
# ==========================================
# Test normalization on intensity anomaly
# ==========================================

test_file = patient_df.loc[
    patient_df["pid"] == "ISPY1_1187", "dce_0"
].iloc[0]

img_1187 = nib.load(test_file)
data_1187 = img_1187.get_fdata()

normalized_1187, lower_1187, upper_1187 = normalize_dce(data_1187)

print("Original:")
print("Min:", data_1187.min())
print("Max:", data_1187.max())

print("\nNormalization limits:")
print("1st percentile:", lower_1187)
print("99th percentile:", upper_1187)

print("\nNormalized:")
print("Min:", normalized_1187.min())
print("Max:", normalized_1187.max())

In [ ]:
# ==========================================
# Check normalization limits across dataset
# ==========================================

normalization_stats = []

for filename in dce_files:

    img = nib.load(os.path.join(dce_path, filename))
    data = img.get_fdata()

    nonzero = data[data > 0]

    lower = np.percentile(nonzero, 1)
    upper = np.percentile(nonzero, 99)

    normalization_stats.append({
        "filename": filename,
        "lower_1pct": lower,
        "upper_99pct": upper
    })

normalization_df = pd.DataFrame(normalization_stats)

print("Volumes checked:", len(normalization_df))

print("\n1st percentile across volumes:")
print(normalization_df["lower_1pct"].describe())

print("\n99th percentile across volumes:")
print(normalization_df["upper_99pct"].describe())

print("\nVolumes with invalid normalization range:",
      (normalization_df["upper_99pct"] <=
       normalization_df["lower_1pct"]).sum())

In [ ]:
# ==========================================
# Compare normalization methods
# ==========================================

def minmax_normalize(data):

    nonzero = data[data > 0]

    lower = nonzero.min()
    upper = nonzero.max()

    normalized = (data - lower) / (upper - lower)
    normalized[data == 0] = 0

    return normalized


def percentile_normalize(data):

    nonzero = data[data > 0]

    lower = np.percentile(nonzero, 1)
    upper = np.percentile(nonzero, 99)

    clipped = np.clip(data, lower, upper)

    normalized = (clipped - lower) / (upper - lower)
    normalized[data == 0] = 0

    return normalized


def zscore_normalize(data):

    nonzero = data[data > 0]

    mean = np.mean(nonzero)
    std = np.std(nonzero)

    normalized = (data - mean) / std
    normalized[data == 0] = 0

    return normalized


# ------------------------------------------
# Load two test patients
# ------------------------------------------

test_patients = ["ISPY1_1219", "ISPY1_1187"]

for patient_id in test_patients:

    file_path = patient_df.loc[
        patient_df["pid"] == patient_id,
        "dce_0"
    ].iloc[0]

    img = nib.load(file_path)
    data = img.get_fdata()

    minmax = minmax_normalize(data)
    percentile = percentile_normalize(data)
    zscore = zscore_normalize(data)

    z = data.shape[0] // 2

    print("\n" + "=" * 50)
    print(patient_id)
    print("=" * 50)

    print("Original range:", data.min(), "to", data.max())

    print("\nMin-max:")
    print(minmax.min(), "to", minmax.max())

    print("\nPercentile:")
    print(percentile.min(), "to", percentile.max())

    print("\nZ-score:")
    print(zscore.min(), "to", zscore.max())

    # Visual comparison
    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(data[z], cmap="gray")
    plt.title("Raw")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(minmax[z], cmap="gray", vmin=0, vmax=1)
    plt.title("Min-Max")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(percentile[z], cmap="gray", vmin=0, vmax=1)
    plt.title("1–99% + Min-Max")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(zscore[z], cmap="gray")
    plt.title("Z-score")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# Test patient-wise common normalization
# ==========================================

def common_patient_normalize(dce_volumes):

    # Collect all non-background voxels
    nonzero_values = np.concatenate([
        volume[volume > 0].ravel()
        for volume in dce_volumes
    ])

    # Common intensity limits for this patient
    lower = np.percentile(nonzero_values, 1)
    upper = np.percentile(nonzero_values, 99)

    normalized_volumes = []

    for volume in dce_volumes:

        # Clip using the SAME limits
        clipped = np.clip(volume, lower, upper)

        # Normalize using the SAME limits
        normalized = (clipped - lower) / (upper - lower)

        # Restore background
        normalized[volume == 0] = 0

        normalized_volumes.append(normalized)

    return normalized_volumes, lower, upper


# ------------------------------------------
# Test on patient 1219
# ------------------------------------------

patient_id = "ISPY1_1219"

dce_volumes = []

for column in ["dce_0", "dce_1", "dce_2"]:

    file_path = patient_df.loc[
        patient_df["pid"] == patient_id,
        column
    ].iloc[0]

    img = nib.load(file_path)
    dce_volumes.append(img.get_fdata())


normalized_volumes, lower, upper = common_patient_normalize(
    dce_volumes
)

print("Patient:", patient_id)

print("\nCommon normalization limits:")
print("1st percentile:", lower)
print("99th percentile:", upper)

print("\nOriginal mean intensities:")

for i, volume in enumerate(dce_volumes):
    print(
        f"DCE {i}:",
        volume[volume > 0].mean()
    )

print("\nNormalized mean intensities:")

for i, volume in enumerate(normalized_volumes):
    print(
        f"DCE {i}:",
        volume[volume > 0].mean()
    )

In [ ]:
# ==========================================
# Test common normalization on anomaly
# ==========================================

patient_id = "ISPY1_1187"

dce_volumes = []

for column in ["dce_0", "dce_1", "dce_2"]:

    file_path = patient_df.loc[
        patient_df["pid"] == patient_id,
        column
    ].iloc[0]

    img = nib.load(file_path)
    dce_volumes.append(img.get_fdata())


normalized_volumes, lower, upper = common_patient_normalize(
    dce_volumes
)

print("Patient:", patient_id)

print("\nCommon normalization limits:")
print("1st percentile:", lower)
print("99th percentile:", upper)

print("\nOriginal mean intensities:")

for i, volume in enumerate(dce_volumes):
    print(
        f"DCE {i}:",
        volume[volume > 0].mean()
    )

print("\nNormalized mean intensities:")

for i, volume in enumerate(normalized_volumes):
    print(
        f"DCE {i}:",
        volume[volume > 0].mean()
    )

In [ ]:
# ==========================================
# Check distribution of volume depth
# ==========================================

depths = []

for _, row in patient_df.iterrows():

    img = nib.load(row["dce_0"])

    depth = img.shape[0]

    depths.append(depth)


depth_df = pd.DataFrame({
    "depth": depths
})

print("Patients checked:", len(depth_df))

print("\nDepth statistics:")
print(depth_df["depth"].describe())

print("\nUnique depths:")
print(sorted(depth_df["depth"].unique()))

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the patient-level master table
master_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/cleaned/patient_master.csv"

patient_df = pd.read_csv(master_path)

print("Total patients:", len(patient_df))
print("Known pCR:", patient_df["pCR"].notna().sum())
print("Missing pCR:", patient_df["pCR"].isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split

# Separate patients with known pCR from the one with missing pCR
labeled_df = patient_df[patient_df["pCR"].notna()].copy()
missing_pcr_df = patient_df[patient_df["pCR"].isna()].copy()

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    labeled_df,
    test_size=0.30,
    stratify=labeled_df["pCR"],
    random_state=42
)

# Second split: divide the remaining 30% equally
# → 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["pCR"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# Check pCR distribution in each split

for name, df in [("Train", train_df),
                 ("Validation", val_df),
                 ("Test", test_df)]:

    print(f"\n{name}")
    print("Total:", len(df))
    print(df["pCR"].value_counts().sort_index())
    print("pCR proportion:")
    print(df["pCR"].value_counts(normalize=True).sort_index())

In [ ]:
# Assign split labels

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Put the patient with missing pCR into training
missing_pcr_df = missing_pcr_df.copy()
missing_pcr_df["split"] = "train"

# Combine everything back together
patient_df = pd.concat(
    [train_df, val_df, test_df, missing_pcr_df],
    axis=0
).sort_index()

print(patient_df["split"].value_counts())

In [ ]:
# Check for patient overlap between splits

train_pids = set(train_df["pid"])
val_pids = set(val_df["pid"])
test_pids = set(test_df["pid"])

print("Train ∩ Validation:", train_pids & val_pids)
print("Train ∩ Test:", train_pids & test_pids)
print("Validation ∩ Test:", val_pids & test_pids)

print("\nTotal unique patients:",
      len(train_pids | val_pids | test_pids | set(missing_pcr_df["pid"])))

In [ ]:
# Save the final patient-level split

master_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/cleaned/patient_master.csv"

patient_df.to_csv(master_path, index=False)

print("Saved successfully!")
print(master_path)

In [ ]:
# Reload the saved table and verify

check_df = pd.read_csv(master_path)

print("Rows:", len(check_df))
print("\nSplit counts:")
print(check_df["split"].value_counts())

print("\nMissing pCR by split:")
print(check_df.groupby("split")["pCR"].apply(lambda x: x.isna().sum()))